# Wound Segmentation Fine-Tuning on FUSeg Dataset

**Goal:** Fine-tune a U-Net (EfficientNet-b3 encoder) on the FUSeg foot ulcer dataset.
**Expected result:** 75-85% Dice score - good enough for a convincing demo.

**Runtime:** Select **GPU (T4)** before running: Runtime -> Change runtime type -> T4 GPU.

> This trains a model for a portfolio/demo project. It is NOT a certified medical device.

## 1. Setup — Install dependencies + clone FUSeg dataset

In [1]:
!pip install -q segmentation-models-pytorch albumentations opencv-python-headless
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121

In [2]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch.nn as nn
import copy
import cv2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
assert device.type == 'cuda', 'GPU required - select T4 runtime'

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
# Clone FUSeg dataset
if not os.path.exists('/content/wound-segmentation'):
    !git clone https://github.com/uwm-bigdata/wound-segmentation.git /content/wound-segmentation

DATA_ROOT = '/content/wound-segmentation/data/Foot Ulcer Segmentation Challenge'
TRAIN_DIR = os.path.join(DATA_ROOT, 'train')
VAL_DIR = os.path.join(DATA_ROOT, 'validation')
TEST_DIR = os.path.join(DATA_ROOT, 'test')

print('Train images:', len(os.listdir(os.path.join(TRAIN_DIR, 'images'))))
print('Val images:', len(os.listdir(os.path.join(VAL_DIR, 'images'))))
print('Test images:', len(os.listdir(os.path.join(TEST_DIR, 'images'))))

## 2. Dataset — PyTorch Dataset + augmentations

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMAGE_SIZE = 512

train_aug = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=20, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05, p=0.3),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

val_aug = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])


class FUSegDataset(Dataset):
    def __init__(self, data_dir, augmentations=None):
        self.img_dir = os.path.join(data_dir, 'images')
        self.mask_dir = os.path.join(data_dir, 'labels')
        self.aug = augmentations
        self.filenames = sorted([
            f for f in os.listdir(self.img_dir)
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ])

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        img = np.array(Image.open(os.path.join(self.img_dir, fname)).convert('RGB'))
        mask = np.array(Image.open(os.path.join(self.mask_dir, fname)).convert('L'))
        mask = (mask > 127).astype(np.float32)  # binary 0/1

        if self.aug:
            augmented = self.aug(image=img, mask=mask)
            img = augmented['image']
            mask = augmented['mask']
        else:
            # fallback if no aug (shouldn't happen, but safe)
            img = torch.from_numpy(img.transpose(2, 0, 1)).float() / 255.0
            mask = torch.from_numpy(mask)

        mask = mask.unsqueeze(0)  # (1, H, W)
        return img, mask


train_ds = FUSegDataset(TRAIN_DIR, augmentations=train_aug)
val_ds = FUSegDataset(VAL_DIR, augmentations=val_aug)

print(f'Train samples: {len(train_ds)}')
print(f'Val samples: {len(val_ds)}')

In [ ]:
# Quick sanity check — visualize one sample
img, mask = train_ds[0]
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(img.permute(1, 2, 0).numpy() * IMAGENET_STD + IMAGENET_MEAN)
axes[0].set_title('Image (denormalized)')
axes[1].imshow(mask.squeeze().numpy(), cmap='gray')
axes[1].set_title('Mask')
plt.show()

## 3. Model — U-Net with EfficientNet-b3 encoder

In [ ]:
model = smp.Unet(
    encoder_name='efficientnet-b3',
    encoder_weights='imagenet',
    in_channels=3,
    classes=1,
    activation=None,
)
model = model.to(device)
print('Model loaded: U-Net + EfficientNet-b3')

## 4. Loss + Metrics — Dice + BCE

In [ ]:
dice_loss = smp.losses.DiceLoss(mode='binary', from_logits=True)
bce_loss = nn.BCEWithLogitsLoss()


def combined_loss(logits, targets):
    return dice_loss(logits, targets) + bce_loss(logits, targets)


def dice_coefficient(pred, target, smooth=1e-6):
    pred = (pred > 0.5).float()
    intersection = (pred * target).sum()
    return (2.0 * intersection + smooth) / (pred.sum() + target.sum() + smooth)


def evaluate(model, loader):
    model.eval()
    total_dice = 0.0
    total_loss = 0.0
    n = 0
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            logits = model(imgs)
            probs = torch.sigmoid(logits)
            total_dice += dice_coefficient(probs, masks).item()
            total_loss += combined_loss(logits, masks).item()
            n += 1
    return total_loss / n, total_dice / n

## 5. Training Loop — 25 epochs, AdamW, ReduceLROnPlateau

In [ ]:
BATCH_SIZE = 8
EPOCHS = 25
LR = 1e-4

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

optimizer = AdamW(model.parameters(), lr=LR)
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

best_dice = 0.0
best_model_wts = None
no_improve_count = 0

for epoch in range(1, EPOCHS + 1):
    # --- Train ---
    model.train()
    train_loss = 0.0
    n_batches = 0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = combined_loss(logits, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        n_batches += 1
    train_loss /= n_batches

    # --- Validate ---
    val_loss, val_dice = evaluate(model, val_loader)

    # --- LR scheduler ---
    scheduler.step(val_dice)

    print(f'Epoch {epoch:2d}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Dice: {val_dice:.4f} | LR: {optimizer.param_groups[0]["lr"]:.2e}')

    # --- Save best ---
    if val_dice > best_dice:
        best_dice = val_dice
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save(best_model_wts, '/content/best_model.pth')
        print(f'  -> New best Dice: {best_dice:.4f} - checkpoint saved')
        no_improve_count = 0
    else:
        no_improve_count += 1

    # --- Early stop if no improvement for 5 epochs ---
    if no_improve_count >= 5:
        print(f'\nVal Dice has not improved for 5 epochs - stopping early at epoch {epoch}')
        print(f'Best Dice: {best_dice:.4f}')
        break

print(f'\nTraining complete. Best validation Dice: {best_dice:.4f}')
print('Best checkpoint saved to /content/best_model.pth')

## 6. Evaluation — Test on 5 unseen images

In [ ]:
# Load best model
model.load_state_dict(torch.load('/content/best_model.pth', map_location=device))
model.eval()

# Get 5 test images
test_img_dir = os.path.join(TEST_DIR, 'images')
test_mask_dir = os.path.join(TEST_DIR, 'labels')
has_labels = os.path.isdir(test_mask_dir)

test_files = sorted([f for f in os.listdir(test_img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])[:5]

ncols = 3 if has_labels else 2
fig, axes = plt.subplots(5, ncols, figsize=(12, 20))
test_dice_scores = []

for i, fname in enumerate(test_files):
    # Load original
    img = np.array(Image.open(os.path.join(test_img_dir, fname)).convert('RGB'))
    img_resized = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))

    # Preprocess
    augmented = val_aug(image=img, mask=np.zeros_like(img[:,:,0]))
    input_tensor = augmented['image'].unsqueeze(0).to(device)

    # Predict
    with torch.no_grad():
        logits = model(input_tensor)
        pred = torch.sigmoid(logits)

    pred_np = pred.squeeze().cpu().numpy()
    pred_bin = (pred_np > 0.5).astype(np.float32)

    col = 0
    axes[i, col].imshow(img_resized)
    axes[i, col].set_title(f'Original: {fname}')
    axes[i, col].axis('off')
    col += 1

    if has_labels:
        gt_path = os.path.join(test_mask_dir, fname)
        if os.path.exists(gt_path):
            gt_mask = np.array(Image.open(gt_path).convert('L'))
            gt_mask_bin = (gt_mask > 127).astype(np.float32)
            augmented_gt = val_aug(image=img, mask=gt_mask_bin)
            gt_tensor = augmented_gt['mask'].unsqueeze(0).to(device)
            dice = dice_coefficient(pred, gt_tensor).item()
            test_dice_scores.append(dice)
            axes[i, col].imshow(gt_tensor.squeeze().cpu().numpy(), cmap='gray')
            axes[i, col].set_title('Ground Truth')
        else:
            axes[i, col].text(0.5, 0.5, 'No label', ha='center', va='center')
            axes[i, col].set_title('Ground Truth (missing)')
        axes[i, col].axis('off')
        col += 1

    axes[i, col].imshow(pred_bin, cmap='gray')
    dice_str = f'Dice: {dice:.3f}' if has_labels and os.path.exists(os.path.join(test_mask_dir, fname)) else ''
    axes[i, col].set_title(f'Predicted {dice_str}')
    axes[i, col].axis('off')

plt.tight_layout()
plt.savefig('/content/test_results.png', dpi=150)
plt.show()

if test_dice_scores:
    print(f'\nTest Dice scores: {[f"{d:.4f}" for d in test_dice_scores]}')
    print(f'Mean test Dice: {np.mean(test_dice_scores):.4f}')
else:
    print('\nNo ground truth labels in test folder - visual inspection only.')

## 7. Export — Download checkpoint

In [ ]:
from google.colab import files
files.download('/content/best_model.pth')

## After downloading — what to do on your machine

1. **Rename** the downloaded file to `wound_unet.pth`

2. **Place it** at:
   ```
   /backend/app/checkpoints/wound_unet.pth
   ```

3. **Set in `.env`:**
   ```
   MODEL_CHECKPOINT_PATH=app/checkpoints/wound_unet.pth
   ```

4. **Verify `model.py` uses `efficientnet-b3`** (not b0) — the encoder must match
   what was trained or you'll get a shape-mismatch error.

5. **Run the self-check:**
   ```bash
   cd backend
   python -m app.model
   ```
   Place a test image in `/backend/samples/` first. Check `samples/output_test.png`.

6. **Test on your own photos** — FUSeg is all foot ulcers under clinical lighting.
   Your phone photos will look different. If the model performs noticeably worse,
   note it honestly in your project's limitations section.